## Similarity Exercise

In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 75.0 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sentence_transformers import SentenceTransformer
import faiss

In this exercise, you've been provided the title and abstract of 500 recent machine learning research papers posted on arXiv.org.

In [4]:
articles = pd.read_csv('./arxiv_papers.csv')
articles.head()

,title,abstract,url
0,GoT-R1: Unleashing Reasoning Capability of MLL...,Visual generation models have made remarkable ...,http://arxiv.org/abs/2505.17022v1
1,Delving into RL for Image Generation with CoT:...,Recent advancements underscore the significant...,http://arxiv.org/abs/2505.17017v1
2,Interactive Post-Training for Vision-Language-...,"We introduce RIPT-VLA, a simple and scalable r...",http://arxiv.org/abs/2505.17016v1
3,When Are Concepts Erased From Diffusion Models?,"Concept erasure, the ability to selectively pr...",http://arxiv.org/abs/2505.17013v1
4,Understanding Prompt Tuning and In-Context Lea...,Prompting is one of the main ways to adapt a p...,http://arxiv.org/abs/2505.17010v1


In [5]:
i = 0
print(f'Title: {articles.loc[i,"title"]}\n')
print(f'Text: {articles.loc[i,"abstract"]}')

Title: GoT-R1: Unleashing Reasoning Capability of MLLM for Visual Generation with Reinforcement Learning

Text: Visual generation models have made remarkable progress in creating realistic
images from text prompts, yet struggle with complex prompts that specify
multiple objects with precise spatial relationships and attributes. Effective
handling of such prompts requires explicit reasoning about the semantic content
and spatial layout. We present GoT-R1, a framework that applies reinforcement
learning to enhance semantic-spatial reasoning in visual generation. Building
upon the Generation Chain-of-Thought approach, GoT-R1 enables models to
autonomously discover effective reasoning strategies beyond predefined
templates through carefully designed reinforcement learning. To achieve this,
we propose a dual-stage multi-dimensional reward framework that leverages MLLMs
to evaluate both the reasoning process and final output, enabling effective
supervision across the entire generation pipeli

Let's try out a variety of ways of vectorizing and searching for semantically-similar papers.

### Method 1: Bag of Words

Fit a CountVectorizer to the abstracts of the articles with all of the defaults.  Then vectorize the dataset using the fit vectorizer.

In [8]:
vectorizer = CountVectorizer().fit(articles['abstract'])
abstracts_vectorized = vectorizer.transform(articles['abstract'])

In [9]:
abstracts_vectorized

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 59465 stored elements and shape (500, 7978)>

**Question:** How many dimensions do the embeddings have?

In [10]:
abstracts_vectorized.shape

(500, 7978)

Now, let's use the embeddings to look for similar articles to a search query.

Apply the vectorizer you fit earlier to this query string to get an embedding.

**Hint:** You can't pass a string to a vectorizer, but you can pass a list containing a string.

In [12]:
query = "vector databases for retrieval augmented generation"

query_vector = vectorizer.transform([query])
# Your code to transform the search query

In [14]:
query = "vector databases for retrieval augmented generation"

query_embedding = vectorizer.transform([query])
query_embedding

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 6 stored elements and shape (1, 7978)>

Now, we need to find the similarity between our query embedding and each vectorized article.

For this, you can use the [cosine similarity function from scikit-learn.](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)

Calculate the similarity between the query embedding and each article embedding and save the result to a variable named `similarity_scores`.

In [15]:
similarity_scores = cosine_similarity(query_embedding, abstracts_vectorized)

In [16]:
articles['abstract'].iloc[8]

'Inverse problems involving differential equations often require identifying\r\nunknown parameters or functions from data. Existing approaches, such as\r\nPhysics-Informed Neural Networks (PINNs), Universal Differential Equations\r\n(UDEs) and Universal Physics-Informed Neural Networks (UPINNs), are effective\r\nat isolating either parameters or functions but can face challenges when\r\napplied simultaneously due to solution non-uniqueness. In this work, we\r\nintroduce a framework that addresses these limitations by establishing\r\nconditions under which unique solutions can be guaranteed. To illustrate, we\r\napply it to examples from biological systems and ecological dynamics,\r\ndemonstrating accurate and interpretable results. Our approach significantly\r\nenhances the potential of machine learning techniques in modeling complex\r\nsystems in science and engineering.'

In [17]:

similarity_scores

array([[0.11020775, 0.08461622, 0.01780047, 0.04145133, 0.03806935,
        0.0814463 , 0.06593805, 0.03782347, 0.        , 0.07059312,
        0.03462717, 0.07968191, 0.03857584, 0.020646  , 0.03197647,
        0.01893206, 0.04279605, 0.03979361, 0.05059374, 0.01539738,
        0.01800706, 0.0184995 , 0.03533326, 0.        , 0.01879115,
        0.03510393, 0.12309149, 0.        , 0.04543109, 0.09829464,
        0.10193473, 0.062177  , 0.02197935, 0.11200358, 0.05493732,
        0.04188539, 0.03503922, 0.06100889, 0.09866713, 0.07699905,
        0.02475369, 0.03209979, 0.05309942, 0.11826248, 0.        ,
        0.02207554, 0.0402259 , 0.09707137, 0.06233787, 0.06225728,
        0.05412659, 0.06384424, 0.        , 0.        , 0.01944039,
        0.1214167 , 0.0285133 , 0.09924856, 0.        , 0.        ,
        0.        , 0.06944891, 0.07385489, 0.0285831 , 0.06463956,
        0.02733833, 0.03914801, 0.0955637 , 0.0843274 , 0.02810497,
        0.04072315, 0.05878964, 0.05236631, 0.09

Now, we need to find the most similar results. To help with this, we can use the [argsort function from numpy](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html), which will give the indices sorted by value.

Use the argsort function to find the indices of the 5 most similar articles. **Warning:** argsort sorts from smallest to largest.

In [19]:
most_similar_articles = np.argsort(-similarity_scores)[0,:5]
most_similar_articles

array([259, 289,  83, 486, 394])

In [21]:
similarity_scores[0, [259, 289,  83, 486, 394]]

array([0.26978155, 0.18926175, 0.18057878, 0.17057206, 0.15681251])

In [20]:
for title, abstract in articles.loc[most_similar_articles, ['title', 'abstract']].values:
    print(title)
    print(abstract)
    print("--------")

MIRB: Mathematical Information Retrieval Benchmark
Mathematical Information Retrieval (MIR) is the task of retrieving
information from mathematical documents and plays a key role in various
applications, including theorem search in mathematical libraries, answer
retrieval on math forums, and premise selection in automated theorem proving.
However, a unified benchmark for evaluating these diverse retrieval tasks has
been lacking. In this paper, we introduce MIRB (Mathematical Information
Retrieval Benchmark) to assess the MIR capabilities of retrieval models. MIRB
includes four tasks: semantic statement retrieval, question-answer retrieval,
premise retrieval, and formula retrieval, spanning a total of 12 datasets. We
evaluate 13 retrieval models on this benchmark and analyze the challenges
inherent to MIR. We hope that MIRB provides a comprehensive framework for
evaluating MIR systems and helps advance the development of more effective
retrieval models tailored to the mathematical domai

Try using a tfidf vectorizer. How do the results compare?

In [23]:
query = "vector databases for retrieval augmented generation"

vectorizer = TfidfVectorizer().fit(articles['abstract'])
abstracts_vectorized = vectorizer.transform(articles['abstract'])
query_embedding = vectorizer.transform([query])
similarity_scores = cosine_similarity(query_embedding, abstracts_vectorized)
most_similar_articles = np.argsort(-similarity_scores)[0,:5]
for title, abstract in articles.loc[most_similar_articles, ['title', 'abstract']].values:
    print(title)
    print(abstract)
    print("--------")

MIRB: Mathematical Information Retrieval Benchmark
Mathematical Information Retrieval (MIR) is the task of retrieving
information from mathematical documents and plays a key role in various
applications, including theorem search in mathematical libraries, answer
retrieval on math forums, and premise selection in automated theorem proving.
However, a unified benchmark for evaluating these diverse retrieval tasks has
been lacking. In this paper, we introduce MIRB (Mathematical Information
Retrieval Benchmark) to assess the MIR capabilities of retrieval models. MIRB
includes four tasks: semantic statement retrieval, question-answer retrieval,
premise retrieval, and formula retrieval, spanning a total of 12 datasets. We
evaluate 13 retrieval models on this benchmark and analyze the challenges
inherent to MIR. We hope that MIRB provides a comprehensive framework for
evaluating MIR systems and helps advance the development of more effective
retrieval models tailored to the mathematical domai

### Method 2: Using a Pretrained Embedding Model

Now, let's compare how we do using the [all-MiniLM-L6-v2 embedding model](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).

This will create a 384-dimensional dense embedding of each sentence.

In [24]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [25]:
sentences = ["This is an example sentence", "Each sentence is converted"]

embeddings = embedder.encode(sentences)
print(embeddings)

[[ 6.76569194e-02  6.34959415e-02  4.87130806e-02  7.93049484e-02
   3.74481045e-02  2.65281182e-03  3.93749401e-02 -7.09844474e-03
   5.93614615e-02  3.15370038e-02  6.00980595e-02 -5.29052690e-02
   4.06068154e-02 -2.59308740e-02  2.98427939e-02  1.12690637e-03
   7.35148638e-02 -5.03818318e-02 -1.22386597e-01  2.37028114e-02
   2.97265425e-02  4.24768664e-02  2.56337449e-02  1.99515442e-03
  -5.69190457e-02 -2.71598250e-02 -3.29035148e-02  6.60249218e-02
   1.19007207e-01 -4.58791330e-02 -7.26214126e-02 -3.25840451e-02
   5.23412861e-02  4.50553298e-02  8.25300720e-03  3.67024392e-02
  -1.39415609e-02  6.53918684e-02 -2.64272150e-02  2.06383236e-04
  -1.36643508e-02 -3.62810828e-02 -1.95043813e-02 -2.89738141e-02
   3.94270159e-02 -8.84090737e-02  2.62426236e-03  1.36713963e-02
   4.83063012e-02 -3.11566219e-02 -1.17329165e-01 -5.11690676e-02
  -8.85288119e-02 -2.18963195e-02  1.42986281e-02  4.44167927e-02
  -1.34815322e-02  7.43392557e-02  2.66382676e-02 -1.98762883e-02
   1.79191

Use this new embedder to vectorize the abstracts and then find the most similar to the query. How do the results compare to the other methods?

**Warning:** Creating embeddings for all of the articles may take a while.

In [26]:
abstracts_vectorized = embedder.encode(articles['abstract'].values)

In [27]:
abstracts_vectorized.shape

(500, 384)

In [28]:
query = "vector databases for retrieval augmented generation"
query_embedding = embedder.encode([query])
similarity_scores = cosine_similarity(query_embedding, abstracts_vectorized)
most_similar_articles = np.argsort(-similarity_scores)[0,:5]
for title, abstract in articles.loc[most_similar_articles, ['title', 'abstract']].values:
    print(title)
    print(abstract)
    print("--------")

MIRB: Mathematical Information Retrieval Benchmark
Mathematical Information Retrieval (MIR) is the task of retrieving
information from mathematical documents and plays a key role in various
applications, including theorem search in mathematical libraries, answer
retrieval on math forums, and premise selection in automated theorem proving.
However, a unified benchmark for evaluating these diverse retrieval tasks has
been lacking. In this paper, we introduce MIRB (Mathematical Information
Retrieval Benchmark) to assess the MIR capabilities of retrieval models. MIRB
includes four tasks: semantic statement retrieval, question-answer retrieval,
premise retrieval, and formula retrieval, spanning a total of 12 datasets. We
evaluate 13 retrieval models on this benchmark and analyze the challenges
inherent to MIR. We hope that MIRB provides a comprehensive framework for
evaluating MIR systems and helps advance the development of more effective
retrieval models tailored to the mathematical domai

### FAISS

The [Faiss library](https://faiss.ai/index.html) is a library for efficient similarity search and clustering of dense vectors. It can be used to automate the process of finding the most similar abstracts.

If we want to use cosine similarity, we need to use the Inner Product. We also need to normalize our vectors so that they all have length 1.

Use the [normalize function](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.normalize.html) to normalize both the abstract vectors and the query vector.

In [30]:
abstracts_vectorized_normalized = normalize(
    abstracts_vectorized
)

In [31]:
(abstracts_vectorized_normalized**2).sum(axis = 1)


array([0.99999994, 1.        , 1.        , 1.0000001 , 1.0000001 ,
       1.        , 1.0000002 , 0.9999999 , 0.99999976, 1.        ,
       1.        , 1.        , 0.9999999 , 1.        , 1.        ,
       1.0000002 , 1.0000002 , 1.        , 0.9999999 , 0.99999976,
       1.        , 1.0000002 , 0.9999999 , 1.0000001 , 1.        ,
       1.        , 1.0000002 , 1.        , 1.        , 1.        ,
       1.        , 0.99999994, 0.9999999 , 1.        , 1.        ,
       1.0000001 , 1.0000002 , 1.        , 1.0000002 , 1.0000001 ,
       1.        , 1.0000001 , 1.0000001 , 1.        , 1.        ,
       0.99999994, 1.0000002 , 1.        , 1.        , 1.0000002 ,
       1.0000002 , 1.        , 1.        , 0.9999999 , 1.        ,
       1.        , 1.        , 1.0000001 , 1.0000001 , 1.        ,
       0.9999999 , 1.0000001 , 1.        , 0.99999994, 0.99999994,
       1.0000002 , 1.        , 1.0000002 , 1.        , 1.0000001 ,
       1.        , 0.9999999 , 1.        , 1.        , 1.     

In [32]:
abstracts_vectorized_normalized = normalize(abstracts_vectorized)
query_embedding_normalized = normalize(query_embedding)

Now, create an [IndexFlatIP object](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes#summary-of-methods) that has dimensions equal to the dimensionality of your vectors. Then add your normalized abstract vectors.

Hint: You can mimic the example [here](https://github.com/facebookresearch/faiss/wiki/Getting-started#building-an-index-and-adding-the-vectors-to-it), but substitute in the IndexFlatIP class.

In [33]:
dimension = abstracts_vectorized.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(abstracts_vectorized_normalized)

Finally, use the [search function](https://github.com/facebookresearch/faiss/wiki/Getting-started#searching) on your index object to find the 5 most similar articles.

In [34]:
index.search(query_embedding_normalized, k=5)

(array([[0.47817042, 0.4554736 , 0.45146084, 0.43434593, 0.4144665 ]],
       dtype=float32),
 array([[259, 107, 233, 209, 247]]))

In [35]:
D, I = index.search(query_embedding_normalized, k=5)
most_similar_articles = I[0]

for title, abstract in articles.loc[most_similar_articles, ['title', 'abstract']].values:
    print(title)
    print(abstract)
    print("--------")

MIRB: Mathematical Information Retrieval Benchmark
Mathematical Information Retrieval (MIR) is the task of retrieving
information from mathematical documents and plays a key role in various
applications, including theorem search in mathematical libraries, answer
retrieval on math forums, and premise selection in automated theorem proving.
However, a unified benchmark for evaluating these diverse retrieval tasks has
been lacking. In this paper, we introduce MIRB (Mathematical Information
Retrieval Benchmark) to assess the MIR capabilities of retrieval models. MIRB
includes four tasks: semantic statement retrieval, question-answer retrieval,
premise retrieval, and formula retrieval, spanning a total of 12 datasets. We
evaluate 13 retrieval models on this benchmark and analyze the challenges
inherent to MIR. We hope that MIRB provides a comprehensive framework for
evaluating MIR systems and helps advance the development of more effective
retrieval models tailored to the mathematical domai